In [3]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

# Dataset with 40 queries across 5 intents
data = [
    ("What can I eat if my blood sugar is high?", "diet_nutrition"),
    ("Is it safe for diabetics to eat bananas?", "diet_nutrition"),
    ("Can I consume sugar-free sweeteners daily?", "diet_nutrition"),
    ("What are the best low glycemic index foods?", "diet_nutrition"),
    ("How many carbohydrates should I eat per meal?", "diet_nutrition"),
    ("Are whole grains good for managing blood glucose?", "diet_nutrition"),
    ("Can I drink fruit juice if I have diabetes?", "diet_nutrition"),
    ("What is a healthy diabetic meal plan?", "diet_nutrition"),
    ("How does fiber help control blood sugar levels?", "diet_nutrition"),
    ("Should I avoid potatoes and white rice?", "diet_nutrition"),

    ("What is a normal fasting blood sugar level?", "blood_glucose"),
    ("How often should I check my glucose with a glucometer?", "blood_glucose"),
    ("Why is my morning blood sugar so high?", "blood_glucose"),
    ("What does HbA1c test measure?", "blood_glucose"),
    ("How do I interpret postprandial glucose numbers?", "blood_glucose"),
    ("What is considered hyperglycemia?", "blood_glucose"),
    ("My blood sugar is 250 mg/dL, is that dangerous?", "blood_glucose"),
    ("How does stress affect my blood glucose readings?", "blood_glucose"),
    ("What is hypoglycemia and what causes it?", "blood_glucose"),
    ("Why do my sugar levels fluctuate throughout the day?", "blood_glucose"),

    ("When should I take my metformin tablet?", "medication_management"),
    ("What are the common side effects of insulin injections?", "medication_management"),
    ("What should I do if I missed my morning diabetes dose?", "medication_management"),
    ("How do I properly store insulin vials?", "medication_management"),
    ("Can I take my diabetes pills on an empty stomach?", "medication_management"),
    ("What is the difference between basal and bolus insulin?", "medication_management"),
    ("Will I need to take medication for the rest of my life?", "medication_management"),
    ("How do GLP-1 receptor agonists help control diabetes?", "medication_management"),
    ("Are there herbal supplements that replace diabetes medications?", "medication_management"),
    ("How do I inject insulin safely using a pen?", "medication_management"),

    ("Can physical activity prevent type 2 diabetes?", "exercise_lifestyle"),
    ("How does cardio workout affect insulin sensitivity?", "exercise_lifestyle"),
    ("Should I check my blood sugar before working out?", "exercise_lifestyle"),
    ("Is strength training safe for diabetic patients?", "exercise_lifestyle"),
    ("How does lack of sleep impact my diabetes control?", "exercise_lifestyle"),
    ("What precautions should I take when exercising outdoors?", "exercise_lifestyle"),
    ("Can yoga help manage diabetes stress?", "exercise_lifestyle"),
    ("How many active minutes per week do I need?", "exercise_lifestyle"),
    ("Why did my blood sugar spike after intense weightlifting?", "exercise_lifestyle"),
    ("What is the best time of day to exercise for blood sugar?", "exercise_lifestyle"),

    ("What is the difference between type 1 and type 2 diabetes?", "general_inquiries"),
    ("Is diabetes a hereditary genetic condition?", "general_inquiries"),
    ("What are the early warning signs of diabetes?", "general_inquiries"),
    ("How does gestational diabetes affect pregnancy?", "general_inquiries"),
    ("Can diabetes lead to nerve damage or neuropathy?", "general_inquiries"),
    ("What doctor specialist should I consult for diabetes care?", "general_inquiries"),
    ("How does uncontrolled diabetes affect kidney function?", "general_inquiries"),
    ("Can type 2 diabetes ever be put into remission?", "general_inquiries"),
    ("What is insulin resistance in simple terms?", "general_inquiries"),
    ("How does diabetes affect cardiovascular health?", "general_inquiries")
]

df = pd.DataFrame(data, columns=["query", "intent"])

# Preprocessing function
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    cleaned = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(cleaned)

df['clean'] = df['query'].apply(clean_text)

# Train test split and TF-IDF
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['intent'], test_size=0.2, random_state=42, stratify=df['intent']
)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

# Evaluation
y_pred = clf.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision, Recall, F1:", precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Knowledge base
kb = {
    "diet_nutrition": "Focus on complex carbs, fiber, and lean proteins. Avoid sugary drinks and white bread.",
    "blood_glucose": "Normal fasting blood sugar is 70-99 mg/dL. Postprandial should be under 140 mg/dL.",
    "medication_management": "Take oral medications with meals. Store unopened insulin in the fridge.",
    "exercise_lifestyle": "Do at least 150 minutes of moderate aerobic activity per week.",
    "general_inquiries": "Diabetes is a chronic condition affecting how your body turns food into energy."
}

# Rules engine
def check_rules(query, intent, confidence):
    q = query.lower()
    
    # 1. Low confidence check first (catches nonsense queries like gibberish)
    if confidence < 0.25:
        return "Low confidence. Please rephrase your question."

    # 2. Emergency check
    if any(w in q for w in ['unconscious', 'fainting', 'chest pain', 'confusion', 'breath smells fruity', 'sugar over 400', 'sugar below 50']):
        return "EMERGENCY: Please seek immediate medical attention or call emergency services."
    
    # 3. Unsupported query check
    if any(w in q for w in ['football', 'weather', 'movie', 'crypto', 'stock market', 'pepperoni pizza']):
        return "Unsupported query. This assistant only answers diabetes-related questions."
        
    # 4. Medication safety rules
    if intent == 'medication_management' and 'double dose' in q:
        return "Warning: Do not double up on missed doses without consulting a doctor."
        
    return None

# Assistant function
def get_response(user_query):
    cleaned = clean_text(user_query)
    vec = vectorizer.transform([cleaned])
    probs = clf.predict_proba(vec)[0]
    idx = np.argmax(probs)
    intent = clf.classes_[idx]
    conf = probs[idx]
    
    print("\nQuery:", user_query)
    print("Predicted Intent:", intent, f"(Confidence: {conf:.2f})")
    
    rule_res = check_rules(user_query, intent, conf)
    if rule_res:
        print("Response:", rule_res)
        return
        
    ans = kb.get(intent, "Sorry, I don't have information on that.")
    print("Response:", ans)

# Testing the required cases
get_response("What are the best low glycemic index foods?")
get_response("Tell me what numbers my glucometer should show in the morning.")
get_response("Can you recommend a good recipe for pepperoni pizza?")
get_response("Xyz abc defgh qwerty")
get_response("I feel very confused and my breath smells fruity, what do I do?")

Accuracy: 0.3
Precision, Recall, F1: (0.13333333333333333, 0.3, 0.18, None)
Confusion Matrix:
 [[0 0 1 0 1]
 [0 0 0 1 1]
 [0 0 0 2 0]
 [0 0 0 2 0]
 [0 0 0 1 1]]

Query: What are the best low glycemic index foods?
Predicted Intent: diet_nutrition (Confidence: 0.35)
Response: Focus on complex carbs, fiber, and lean proteins. Avoid sugary drinks and white bread.

Query: Tell me what numbers my glucometer should show in the morning.
Predicted Intent: blood_glucose (Confidence: 0.35)
Response: Normal fasting blood sugar is 70-99 mg/dL. Postprandial should be under 140 mg/dL.

Query: Can you recommend a good recipe for pepperoni pizza?
Predicted Intent: diet_nutrition (Confidence: 0.26)
Response: Unsupported query. This assistant only answers diabetes-related questions.

Query: Xyz abc defgh qwerty
Predicted Intent: medication_management (Confidence: 0.20)
Response: Low confidence. Please rephrase your question.

Query: I feel very confused and my breath smells fruity, what do I do?
Predicte

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sujalgupta/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sujalgupta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sujalgupta/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sujalgupta/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
